# MiniLM embedding-space ablations

Test embedding-space scoring variants for **`sentence-transformers/all-MiniLM-L6-v2`** with fixed datasets, evaluation metrics, and cosine-based retrieval.

The hypothesis is that many embedding dimensions contribute non-query-specific background or noise, whereas dimensions in which the query is unusual relative to the document distribution should receive more weight.

Evaluated variants:
1. standard cosine
2. mean-centered cosine
3. variance-normalized cosine
4. z-normalized cosine
5. query-adapted weighted cosine after z-normalization


In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    else:
        !git -C {REPO_ROOT} pull --ff-only
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
!pip -q install -U datasets sentence-transformers


In [ ]:
import hashlib
import logging
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer
from src.embedding_transforms import EmbeddingTransformConfig, EmbeddingTransformType
from src.evaluate import RuntimeConfig, compute_calibration_statistics, evaluate, register_evaluation, register_pipeline
from src.io import load_calibration_set, mount_google_drive
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logger = logging.getLogger("minilm-ablation")


## Shared configuration

Calibration uses a fixed document corpus disjoint from evaluation datasets. `mu` and `sigma` are estimated dimension-wise from raw, non-L2-normalized document embeddings.


In [ ]:
DATASETS_ROOT = "/content/drive/MyDrive/Retreaval/data"
CALIBRATION_SET_PATH = "/content/drive/MyDrive/Retreaval/calibration/bioasq-5k"
REGISTRY_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/datasets.sqlite"
RESULTS_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/results.sqlite"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EPSILON = 1e-6
QUERY_WEIGHT_ALPHA = 1.0
METRIC_CONFIG = {
    "mrr_at_k": (10,),
    "ndcg_at_k": (10,),
    "accuracy_at_k": (1, 3, 5, 10, 100),
    "precision_recall_at_k": (1, 3, 5, 10, 100),
    "map_at_k": (100,),
}
runtime = RuntimeConfig(batch_size=64, corpus_scan_size=10_000, show_progress_bar=True, device="cuda" if torch.cuda.is_available() else "cpu")
evaluation_id = register_evaluation(METRIC_CONFIG, registry_db_path=REGISTRY_DB_PATH)
logger.info("Evaluation: %s | device: %s", evaluation_id, runtime.device)


## Calibration statistics

The calibration source ID hashes the actual document IDs and text. The full `mu` and `sigma` vectors are stored in pipeline identity as well.


In [ ]:
def calibration_source_id(corpus: dict[str, str]) -> str:
    digest = hashlib.sha256()
    for document_id in sorted(corpus):
        for value in (document_id, corpus[document_id]):
            encoded = value.encode("utf-8")
            digest.update(len(encoded).to_bytes(8, byteorder="big"))
            digest.update(encoded)
    return f"bioasq-5k:{digest.hexdigest()[:24]}"
mount_google_drive()
calibration_set = load_calibration_set(CALIBRATION_SET_PATH)
source_id = calibration_source_id(calibration_set.corpus)
calibration_model = SentenceTransformer(MODEL_NAME, device=runtime.device)
calibration_statistics = compute_calibration_statistics(
    calibration_model, calibration_set.corpus.values(), source_id=source_id,
    batch_size=runtime.batch_size, show_progress_bar=runtime.show_progress_bar,
)
logger.info("Calibration source: %s | dimensions: %d", source_id, len(calibration_statistics.mean))


## Register ablation pipelines

For mean centering, variance normalization, and z-normalization, cosine normalization is applied after the corpus-wide transform.

For query adaptation, start from z-normalized embeddings and use `w_k = |z_q,k|^alpha` in a weighted cosine. `alpha=0` is equivalent to z-normalized cosine; this notebook uses `alpha=1` as the first query-adapted condition.


In [ ]:
def calibrated_transform(transform_type, *, alpha=1.0):
    return EmbeddingTransformConfig(
        transform_type=transform_type, calibration=calibration_statistics,
        epsilon=EPSILON, alpha=alpha,
    )
identity_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", registry_db_path=REGISTRY_DB_PATH)
mean_center_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", embedding_transform=calibrated_transform(EmbeddingTransformType.MEAN_CENTER), registry_db_path=REGISTRY_DB_PATH)
variance_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", embedding_transform=calibrated_transform(EmbeddingTransformType.VARIANCE_NORMALIZE), registry_db_path=REGISTRY_DB_PATH)
z_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", embedding_transform=calibrated_transform(EmbeddingTransformType.Z_NORMALIZE), registry_db_path=REGISTRY_DB_PATH)
query_adapted_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", embedding_transform=calibrated_transform(EmbeddingTransformType.QUERY_ADAPTED_Z, alpha=QUERY_WEIGHT_ALPHA), registry_db_path=REGISTRY_DB_PATH)
pipelines = {
    "MiniLM / cosine": identity_pipeline_id,
    "MiniLM / mean-center + cosine": mean_center_pipeline_id,
    "MiniLM / variance-normalize + cosine": variance_pipeline_id,
    "MiniLM / z-normalize + cosine": z_pipeline_id,
    f"MiniLM / query-adapted z + weighted cosine (alpha={QUERY_WEIGHT_ALPHA:g})": query_adapted_pipeline_id,
}
pipelines


## Evaluate every latest dataset version

All variants share the same model, datasets, and `EvaluationDefinition`; only the embedding/scoring geometry changes.


In [ ]:
records = []
for pipeline_name, pipeline_id in pipelines.items():
    outcomes = evaluate(
        pipeline_id=pipeline_id, evaluation_id=evaluation_id, datasets_root=DATASETS_ROOT,
        runtime=runtime, registry_db_path=REGISTRY_DB_PATH, results_db_path=RESULTS_DB_PATH,
    )
    for outcome in outcomes:
        record = {"pipeline": pipeline_name, "dataset": outcome.dataset_name, "version": outcome.dataset_version, "status": outcome.status.value, "dataset_id": outcome.dataset_id, "result_id": outcome.result_id}
        if outcome.metrics is not None:
            record.update(outcome.metrics)
        records.append(record)
results_table = pd.DataFrame(records).sort_values(["pipeline", "dataset", "version"])
identifier_columns = {"pipeline", "dataset", "version", "status", "dataset_id", "result_id"}
metric_columns = [column for column in results_table.columns if column not in identifier_columns]
display(results_table.style.format({column: "{:.4f}" for column in metric_columns}, na_rep=""))


## Follow-up: top-k dimension gating

Rank dimensions independently per query by `|z_q,k|` and retain only the most query-specific dimensions before scoring. Initial gates: **25%, 50%, 75%**. A broader sweep of **5%, 10%, 25%, 50%, 75%, 100%** would directly test whether retrieval quality remains stable or improves while dimensions are removed.

This remains a TODO because it requires a query-dependent sparse scoring mode rather than another corpus-wide transform.
